# Deep Dive: PyTorch `autograd.grad` vs. `autograd.backward`

PyTorch is a popular deep learning framework that provides automatic differentiation through its **autograd** module. This module is essential for training neural networks, as it automates the computation of gradients—a process crucial for optimization algorithms like Gradient Descent and Adam.

Within the `torch.autograd` module, two primary functions are used for computing gradients:
1. `torch.autograd.backward` (or `.backward()` on a Tensor)
2. `torch.autograd.grad`

Although both serve the fundamental purpose of evaluating vector-Jacobian products, they differ significantly in their interfaces, return values, memory management, and practical use cases. This article explores these differences in detail.

---

## 1. High-Level Overview & Key Differences

| Feature / Behavior | `torch.autograd.backward` / `tensor.backward()` | `torch.autograd.grad` |
| :--- | :--- | :--- |
| **Primary Use Case** | Standard neural network training loop (backpropagation). | Higher-order derivatives, custom gradient control, physics-informed neural networks (PINNs). |
| **Return Value** | `None` (modifies `.grad` attributes in-place). | Returns a `tuple` of computed gradient tensors. |
| **Gradient Accumulation** | Accumulates (adds) results into leaf tensors' `.grad` attributes. | Does **not** accumulate gradients into `.grad` attributes; returns fresh tensors. |
| **Computation Graph** | Clears the graph by default after `backward()` (unless `retain_graph=True`). | Clears the graph by default unless `retain_graph=True` or `create_graph=True`. |
| **Higher-Order Gradients** | Requires `create_graph=True`, but accumulative logic makes higher derivatives tedious. | Designed for higher-order derivatives (`create_graph=True` builds a graph of the gradient computation itself). |

---

## 2. Deep Dive: `torch.autograd.backward` / `.backward()`

### 2.1 Mechanism
When you call `loss.backward()`, PyTorch computes the gradient of `loss` with respect to all leaf nodes in the computation graph that have `requires_grad=True`.

Instead of returning the computed gradients directly to the caller, `.backward()` populates or updates the `.grad` attribute of each leaf tensor in-place.

```python
import torch

x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x[0]**2 + x[1]**3

# Perform backpropagation
y.backward()

print(x.grad)  # tensor([4.0, 27.0])
```

### 2.2 Gradient Accumulation
A critical feature (and potential pitfall) of `.backward()` is that it **accumulates** gradients into the `.grad` field using addition (`+`).

```python
# If backward is called again without zeroing gradients:
# y = x[0]^2 + x[1]^3
y = x[0]**2 + x[1]**3
y.backward()

print(x.grad)  # tensor([8.0, 54.0]) -> Gradients accumulated!
```

Because of this behavior, standard PyTorch training loops require explicitly resetting gradients via `optimizer.zero_grad()` or `model.zero_grad()`.

---

## 3. Deep Dive: `torch.autograd.grad`

### 3.1 Mechanism
Unlike `.backward()`, `torch.autograd.grad` computes and **directly returns** the sum of gradients of outputs with respect to inputs. It does not update the `.grad` attributes of the input tensors.

```python
import torch

x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x[0]**2 + x[1]**3

# Compute gradient explicitly
grads = torch.autograd.grad(outputs=y, inputs=x)

print(grads)      # (tensor([4.0, 27.0]),)
print(x.grad)     # None (x.grad remains untouched!)
```

### 3.2 Key Arguments
* **`outputs`**: Tensor or sequence of Tensors to be differentiated.
* **`inputs`**: Sequence of Tensors with respect to which gradients are computed.
* **`grad_outputs`**: Vector for the vector-Jacobian product (defaults to `torch.ones_like(outputs)` for scalar outputs).
* **`retain_graph`**: Keeps the computation graph intact for further differentiation.
* **`create_graph`**: Constructs a derivative graph, enabling higher-order derivatives (e.g., computing Hessian-vector products).

---

## 4. When to Use Which?

### Use `autograd.backward` when:
* You are building standard deep learning training pipelines.
* You are using built-in PyTorch optimizers (`torch.optim.SGD`, `torch.optim.Adam`), which expect leaf tensors to have populated `.grad` fields.
* You want to aggregate gradients over multiple mini-batches before executing an optimization step.

### Use `autograd.grad` when:
* You need **higher-order derivatives** (e.g., computing $f''(x)$, Hessians, or Jacobian-vector products).
* You are implementing **Physics-Informed Neural Networks (PINNs)** where the loss function explicitly depends on derivatives of outputs with respect to inputs (e.g., $rac{\partial u}{\partial t} + u rac{\partial u}{\partial x} = 0$).
* You are implementing specialized GAN losses (e.g., WGAN-GP gradient penalty).
* You need to compute gradients for non-leaf intermediate tensors without altering the `.grad` state of leaf nodes.

---

## 5. Summary Code Comparison

```python
import torch

# Setup
w = torch.tensor([3.0], requires_grad=True)
x = torch.tensor([2.0])
y = w * x

# ------------------------------------
# Method A: autograd.backward / .backward()
# ------------------------------------
y.backward(retain_graph=True)
print("backward() result in w.grad:", w.grad) # tensor([2.0])

# ------------------------------------
# Method B: autograd.grad
# ------------------------------------
grad_result = torch.autograd.grad(outputs=y, inputs=w)
print("autograd.grad result returned:", grad_result[0]) # tensor([2.0])
```

In [57]:
import torch

In [58]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x[0]**2 + x[1]**3

# Compute gradient explicitly
grads = torch.autograd.grad(outputs=y, inputs=x)

print(grads)      # (tensor([4.0, 27.0]),)
print(x.grad)     # None (x.grad remains untouched!)

(tensor([ 4., 27.]),)
None


In [59]:

# Setup
w = torch.tensor([3.0], requires_grad=True)
x = torch.tensor([2.0])
y = w * x

# ------------------------------------
# Method A: autograd.backward / .backward()
# ------------------------------------
y.backward(retain_graph=True)
print("backward() result in w.grad:", w.grad) # tensor([2.0])

# ------------------------------------
# Method B: autograd.grad
# ------------------------------------
grad_result = torch.autograd.grad(outputs=y, inputs=w)
print("autograd.grad result returned:", grad_result[0]) # tensor([2.0])

backward() result in w.grad: tensor([2.])
autograd.grad result returned: tensor([2.])


In [60]:
x = torch.tensor(3.0, requires_grad=True,retain_graph=True)

In [61]:
y = x**2

In [62]:
x

tensor(3., requires_grad=True)

In [63]:
y

tensor(9., grad_fn=<PowBackward0>)

In [64]:
y.backward()

In [65]:
x.grad

tensor(6.)

In [66]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x**2)

In [67]:
dz_dx(4)

-7.661275842587077

In [68]:
x = torch.tensor(4.0, requires_grad=True)

In [69]:
y = x ** 2

In [70]:
z = torch.sin(y)

In [71]:
x

tensor(4., requires_grad=True)

In [72]:
y

tensor(16., grad_fn=<PowBackward0>)

In [73]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [74]:
z.backward()

In [75]:
x.grad

tensor(-7.6613)

In [76]:
y.grad

/tmp/ipykernel_1465/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


In [77]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [78]:
def binary_cross_entropy_loss(prediction, target):

    # Small value to avoid numerical problems
    # Why? Because log(0) is undefined (-infinity)
    # So we never want prediction to become exactly 0 or exactly 1
    epsilon = 1e-8

    # Clamp means: force values to stay inside a safe range
    # If prediction < epsilon → set it to epsilon
    # If prediction > (1 - epsilon) → set it to (1 - epsilon)
    # This prevents log(0) when computing:
    # log(prediction) or log(1 - prediction)

    prediction = torch.clamp(prediction, epsilon, 1 - epsilon) # torch.clamp(x, min_value, max_value)

    # BCE formula:
    # target * log(prediction)     → used when target = 1
    # (1 - target) * log(1 - prediction) → used when target = 0
    # Negative sign makes loss positive

    loss = -(target * torch.log(prediction) +
             (1 - target) * torch.log(1 - prediction))

    return loss

In [79]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [80]:
loss

tensor(6.7012)

In [81]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [82]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [83]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [84]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [85]:
w

tensor(1., requires_grad=True)

In [86]:
b

tensor(0., requires_grad=True)

In [87]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [88]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [89]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [90]:
loss.backward()

In [91]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [92]:
# for multiple

In [93]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [94]:
x

tensor([1., 2., 3.], requires_grad=True)

In [95]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [96]:
y.backward()

In [97]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

In [98]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [99]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [100]:
y.backward()

In [101]:
x.grad

tensor(4.)

In [102]:
x.grad.zero_()

tensor(0.)

In [103]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [104]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [105]:
y.backward()

In [106]:
x.grad

tensor(4.)

In [107]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [108]:
x.requires_grad_(False)

tensor(2.)

In [109]:
x

tensor(2.)

In [110]:
y = x ** 2

In [111]:
y

tensor(4.)

In [112]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
z = x.detach()
z

In [ ]:
y = x ** 2

In [ ]:
y

In [ ]:
y1 = z ** 2
y1

In [ ]:
y.backward()

In [ ]:
y1.backward()

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
x

In [ ]:
y = x ** 2

In [ ]:
y

In [ ]:
y.backward()